# Coarse-grained RNA simulations with CALVADOS — a hands-on tutorial

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anshuJS/V21-RNA-simulation-CALVADOS-tutorial/blob/main/V21_RNA_CALVADOS_tutorial.ipynb)

Welcome! This notebook teaches **molecular dynamics (MD)** and **coarse-grained (CG)** modelling of RNA through one concrete example: a 21-nucleotide RNA we call **V21** (`UAUAAUUAAUAACUAAUUACU`).

You will:
1. **Learn the ideas** — what MD is, what coarse-graining buys you, and the CALVADOS force field — with equations and live **animations**.
2. **Set up a simulation** — step by step, *visualising what we build at every stage* (we configure the system but do **not** run a long simulation here — that needs a GPU cluster).
3. **Analyse real trajectories** — pre-computed CALVADOS runs (shipped in this repo) of V21 across a salt series, measuring its **radius of gyration** and **end-to-end distance**.

> Runs on a **free Colab CPU runtime**. Just run the cells top to bottom.


---
# Part 1 — The ideas

## 1.1 What is a molecular dynamics simulation?

MD computes how a molecule moves by integrating **Newton's equations of motion** for every particle. The force on particle $i$ is the negative gradient of a potential energy function $U$ (the *force field*):

$$\mathbf{F}_i = m_i\,\mathbf{a}_i = -\nabla_i\, U(\mathbf{r}_1,\dots,\mathbf{r}_N)$$

Given positions and velocities now, we step forward by a tiny timestep $\Delta t$ using the **velocity-Verlet** integrator:

$$\mathbf{r}(t+\Delta t) = \mathbf{r}(t) + \mathbf{v}(t)\,\Delta t + \tfrac{1}{2}\mathbf{a}(t)\,\Delta t^2,\qquad
\mathbf{v}(t+\Delta t) = \mathbf{v}(t) + \tfrac{1}{2}\big[\mathbf{a}(t)+\mathbf{a}(t+\Delta t)\big]\Delta t$$

Repeating this millions of times produces a **trajectory** — a movie of the molecule exploring its shapes. Everything physical (how compact the RNA is, how it responds to salt) is then a *statistical average* over that movie.

The animation below shows the simplest possible case: **one bead in a harmonic energy well**, the motion produced by $\mathbf{F}=-\nabla U$. This is exactly what a *bond* does in a force field.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

k, x0 = 5.0, 0.0                          # spring constant, equilibrium position
xs = np.linspace(-2, 2, 300)
U  = 0.5 * k * (xs - x0)**2               # harmonic potential U = 1/2 k (x-x0)^2

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot(xs, U, color="steelblue", lw=2)
ax.set_xlabel("bead position $x$"); ax.set_ylabel("energy $U(x)$")
ax.set_title("a bead oscillating in a harmonic well  ($F=-dU/dx$)")
bead, = ax.plot([], [], "o", color="crimson", ms=16)
A, omega = 1.6, 2.2
def upd(i):
    t = i * 0.08; xb = A * np.cos(omega * t)     # analytic SHM solution
    bead.set_data([xb], [0.5 * k * (xb - x0)**2])
    return (bead,)
anim = animation.FuncAnimation(fig, upd, frames=70, interval=70, blit=True)
plt.close(fig); HTML(anim.to_jshtml())

## A more realistic interaction: the Lennard-Jones potential

A harmonic well is perfect for a **bond** (two beads tied together). But two beads that are *not* bonded behave differently — they can't overlap (hard repulsion when too close) and weakly attract at medium range (van der Waals). The **Lennard-Jones (LJ)** potential captures both:

$$U_\text{LJ}(r)=4\varepsilon\left[\left(\frac{\sigma}{r}\right)^{12}-\left(\frac{\sigma}{r}\right)^{6}\right]$$

The $r^{-12}$ term is the repulsive **wall** (excluded volume — beads have a size $\sigma$); the $r^{-6}$ term is the attractive tail; they balance at the **minimum** $r=2^{1/6}\sigma$ with well-depth $\varepsilon$. This is the backbone of CALVADOS's nonbonded **Ashbaugh–Hatch** term (Part 1.4) — which simply adds a knob $\lambda$ to scale the attraction. Watch a free bead fall in and settle at the equilibrium separation.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

sig, eps = 0.62, 1.0
rmin = 2**(1/6) * sig
rr = np.linspace(0.55, 2.2, 400)
Ulj = 4*eps*((sig/rr)**12 - (sig/rr)**6)

# damped descent: a free bead approaches from far, bounces off the wall, settles at the minimum
def Flj(r):
    f = 24*eps*(2*(sig/r)**12 - (sig/r)**6) / r        # -dU/dr, capped for stability
    return max(min(f, 120.0), -120.0)
pos, vel, dt, gamma, traj = 2.0*sig, -1.5, 1.2e-3, 1.8, []
for _ in range(1800):
    a = Flj(pos) - gamma*vel
    vel += a*dt; pos += vel*dt
    pos = min(max(pos, 0.92*sig), 2.3); traj.append(pos)
traj = traj[::20]

fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.4), gridspec_kw={"width_ratios":[1.5,1]})
ax[0].plot(rr, Ulj, color="teal", lw=2); ax[0].axhline(0, color="gray", lw=0.6)
ax[0].axvline(rmin, color="gray", ls=":", lw=1); ax[0].set_ylim(-1.5, 3)
ax[0].set_xlabel("distance $r$ (nm)"); ax[0].set_ylabel("$U_{LJ}$")
ax[0].set_title("Lennard-Jones: repulsion + attraction")
dot, = ax[0].plot([], [], "o", color="crimson", ms=12)
ax[1].set_xlim(-0.2, 2.3); ax[1].set_ylim(-1, 1); ax[1].set_xticks([]); ax[1].set_yticks([])
ax[1].set_title("two non-bonded beads")
ax[1].plot([0], [0], "o", color="royalblue", ms=26)
b1, = ax[1].plot([], [], "o", color="crimson", ms=26)
def upd(i):
    r = traj[i % len(traj)]
    dot.set_data([r], [4*eps*((sig/r)**12 - (sig/r)**6)]); b1.set_data([r], [0])
    return dot, b1
anim = animation.FuncAnimation(fig, upd, frames=len(traj), interval=55, blit=True)
plt.close(fig); HTML(anim.to_jshtml())

## Cutoffs: how short-range and long-range forces are computed

A system of $N$ beads has up to $N^2$ nonbonded pairs — evaluating every pair every step is expensive. The standard trick is a **cutoff radius** $r_\text{cut}$: compute interactions only for pairs closer than $r_\text{cut}$, and treat everything beyond as **zero**.

This is safe for **short-range** potentials that decay fast: the LJ / Ashbaugh–Hatch attraction ($\sim r^{-6}$) is essentially zero past ~2–3 $\sigma$, so a modest cutoff loses almost nothing. But **electrostatics** ($\sim 1/r$, the Debye–Hückel term) is **long-range** — it still matters well beyond any practical cutoff, so it needs a larger cutoff or special handling. (Salt screening, coming up in 1.4, is exactly what makes electrostatics *effectively* short-ranged again — letting CALVADOS use a finite cutoff.) The animation sweeps $r_\text{cut}$ and shows how much of each potential is captured.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

sig, eps, lB = 0.62, 1.0, 0.71
r = np.linspace(0.55, 5.0, 600)
U_short = 4*eps*((sig/r)**12 - (sig/r)**6)        # LJ — short-range (van der Waals)
U_long  = lB / r                                   # bare Coulomb — long-range

fig, ax = plt.subplots(figsize=(6.8, 3.7))
ax.plot(r, U_short, color="teal",   lw=1, alpha=0.30)
ax.plot(r, U_long,  color="purple", lw=1, alpha=0.30)
ls, = ax.plot([], [], color="teal",   lw=2.6, label="LJ (short-range)")
ll, = ax.plot([], [], color="purple", lw=2.6, label="electrostatic (long-range)")
vline = ax.axvline(0, color="red", ls="--", lw=1.5)
ax.set_ylim(-1.2, 2.6); ax.set_xlabel("distance $r$ (nm)"); ax.set_ylabel("$U$")
ax.axhline(0, color="gray", lw=0.5); ax.legend(loc="upper right", fontsize=8)
txt = ax.text(0.03, 0.96, "", transform=ax.transAxes, va="top", fontsize=9)
tot_s = np.trapz(np.abs(U_short), r); tot_l = np.trapz(np.abs(U_long), r)
cuts = np.linspace(0.9, 4.6, 48)
def upd(i):
    rc = cuts[i % len(cuts)]; m = r <= rc
    ls.set_data(r[m], U_short[m]); ll.set_data(r[m], U_long[m]); vline.set_xdata([rc, rc])
    fs = np.trapz(np.abs(U_short[m]), r[m]) / tot_s
    fl = np.trapz(np.abs(U_long[m]),  r[m]) / tot_l
    txt.set_text(f"cutoff = {rc:.1f} nm\nLJ captured: {fs*100:3.0f}%\nelectrostatic captured: {fl*100:3.0f}%")
    ax.set_title("beyond the cutoff, the interaction is set to 0 (skipped)")
    return ls, ll, vline, txt
anim = animation.FuncAnimation(fig, upd, frames=len(cuts), interval=110, blit=False)
plt.close(fig); HTML(anim.to_jshtml())

## 1.2 Why coarse-graining?

A fully **atomistic** simulation tracks every atom (and often every water molecule). That is accurate but brutally expensive: biological motions of an RNA or protein happen over micro- to milliseconds, while an atomistic timestep is ~1 femtosecond ($10^{-15}$ s) — billions of steps.

**Coarse-graining** groups atoms into a few **beads**, drastically cutting the number of particles and letting us use a bigger timestep and an *implicit* (invisible) solvent. We trade atomic detail for the ability to reach long timescales and large assemblies.

| | all-atom | coarse-grained (CALVADOS) |
|---|---|---|
| RNA nucleotide | ~30 atoms | **2 beads** (phosphate + base) |
| solvent | explicit water | implicit (Debye–Hückel) |
| timestep | ~1–2 fs | ~10 fs |
| reach | ns–µs, single molecule | µs–ms, many molecules |

The schematic below contrasts the two representations for a few nucleotides.

In [ ]:
import matplotlib.pyplot as plt, numpy as np
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
rng = np.random.default_rng(0)
# all-atom cartoon: many small dots
for n in range(4):
    c = np.array([n*1.5, 0]); pts = c + rng.normal(0, 0.28, (28, 2))
    axes[0].scatter(pts[:,0], pts[:,1], s=12, color="gray")
axes[0].set_title("all-atom (~30 atoms / nucleotide)")
# CG: 2 beads per nucleotide
for n in range(4):
    axes[1].scatter([n*1.5], [0.35], s=420, color="#e377c2")          # phosphate
    axes[1].scatter([n*1.5], [-0.35], s=420, color="#2ca02c")         # base
    if n: axes[1].plot([(n-1)*1.5, n*1.5], [0.35,0.35], "k-", lw=1)   # backbone
    axes[1].plot([n*1.5, n*1.5], [0.35,-0.35], "k-", lw=1)
axes[1].scatter([], [], s=120, color="#e377c2", label="phosphate bead")
axes[1].scatter([], [], s=120, color="#2ca02c", label="base bead")
axes[1].legend(loc="upper center", ncol=2, fontsize=8); axes[1].set_title("CALVADOS RNA (2 beads / nucleotide)")
for a in axes: a.set_xticks([]); a.set_yticks([]); a.set_ylim(-1.2,1.2)
plt.tight_layout(); plt.show()

## 1.3 What CALVADOS does

**CALVADOS** is a residue-level CG model originally for intrinsically disordered proteins, extended to RNA. Each bead carries:
- a **size** $\sigma$, a **charge** $q$, and a **"stickiness"** $\lambda$ (how much it likes to touch other beads — derived from hydropathy).

The total energy is a sum of simple, physically interpretable terms:

$$U = \underbrace{U_\text{bond} + U_\text{angle}}_{\text{connectivity}} \;+\; \underbrace{U_\text{AH}}_{\text{stickiness / excluded volume}} \;+\; \underbrace{U_\text{DH}}_{\text{electrostatics (salt!)}}$$

For RNA specifically: **2 beads per nucleotide** (phosphate + base), backbone bonds, an angle term, nearest-neighbour base **stacking**, and the nonbonded terms above. Importantly, the model has **no Watson–Crick base-pairing built in** and **no base-specific A/U/C/G parameters** — base identity only sets bead size. (This matters later: to make a *hairpin* we must add restraints ourselves.)

The next two cells animate the two nonbonded terms — they are the heart of the model.

### The Ashbaugh–Hatch term $U_\text{AH}$ — stickiness & excluded volume

$$U_\text{AH}(r)=\begin{cases} U_\text{LJ}(r) + (1-\lambda)\,\varepsilon, & r < 2^{1/6}\sigma\\[4pt] \lambda\,U_\text{LJ}(r), & r \ge 2^{1/6}\sigma\end{cases}\qquad U_\text{LJ}(r)=4\varepsilon\!\left[\left(\tfrac{\sigma}{r}\right)^{12}-\left(\tfrac{\sigma}{r}\right)^{6}\right]$$

The short-range part is always **repulsive** (beads can't overlap — excluded volume). The parameter $\lambda$ scales the **attractive** tail: high $\lambda$ → sticky (collapses), low $\lambda$ → barely attractive (expands). Watch the attractive well deepen as $\lambda$ increases.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

sig, eps = 0.62, 0.8368                      # nm, kJ/mol (CALVADOS-like)
r = np.linspace(0.55, 2.0, 400)
def U_AH(r, lam):
    rmin = 2**(1/6)*sig; lj = 4*eps*((sig/r)**12 - (sig/r)**6)
    return np.where(r < rmin, lj + (1-lam)*eps, lam*lj)

fig, ax = plt.subplots(figsize=(5.5, 3.4))
line, = ax.plot([], [], lw=2, color="darkgreen")
ax.set_xlim(0.55, 2.0); ax.set_ylim(-1.0, 2.0)
ax.axhline(0, color="gray", lw=0.6)
ax.set_xlabel("bead–bead distance $r$ (nm)"); ax.set_ylabel("$U_{AH}$ (kJ/mol)")
txt = ax.text(0.97, 0.92, "", transform=ax.transAxes, ha="right")
lams = np.linspace(0.0, 1.18, 60)
def upd(i):
    lam = lams[i % len(lams)]
    line.set_data(r, U_AH(r, lam)); txt.set_text(f"stickiness λ = {lam:.2f}")
    ax.set_title("Ashbaugh–Hatch: λ controls attraction"); return line, txt
anim = animation.FuncAnimation(fig, upd, frames=len(lams), interval=80, blit=True)
plt.close(fig); HTML(anim.to_jshtml())

### The Debye–Hückel term $U_\text{DH}$ — electrostatics and **salt screening**

RNA is highly charged (each phosphate carries $-1$). In an ionic solution, dissolved salt **screens** these charges. CALVADOS uses a Debye–Hückel potential:

$$U_\text{DH}(r) = \frac{q_i q_j}{4\pi\varepsilon_0\varepsilon_r}\,\frac{e^{-\kappa r}}{r},\qquad \kappa = \sqrt{\frac{2 N_A e^2 I}{\varepsilon_0\varepsilon_r k_B T}}$$

where $I$ is the **ionic strength** (salt concentration) and $1/\kappa$ is the **Debye length** (how far a charge is "felt"). More salt → larger $\kappa$ → shorter Debye length → charges screened at shorter range. The animation shows the repulsion between two phosphates collapsing as salt rises — **this is exactly why the RNA compacts with salt** in Part 3.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

lB = 0.71            # Bjerrum length (nm) in water at 300 K
r  = np.linspace(0.6, 6.0, 400)
def kappa(I_molar):  # 1/nm ; I in mol/L
    return np.sqrt(8*np.pi*lB*I_molar*6.022e23/1e24) if I_molar > 0 else 1e-6
def U_DH(r, I):
    return lB*np.exp(-kappa(I)*r)/r          # units of kB*T per unit charge^2

salts = [0, 10, 20, 50, 100, 150, 300, 500, 1000]   # mM
fig, ax = plt.subplots(figsize=(5.5, 3.4))
line, = ax.plot([], [], lw=2, color="purple")
ax.set_xlim(0.6, 6.0); ax.set_ylim(0, 1.2)
ax.set_xlabel("phosphate–phosphate distance $r$ (nm)"); ax.set_ylabel("$U_{DH}$  ($k_BT$)")
txt = ax.text(0.97, 0.9, "", transform=ax.transAxes, ha="right")
def upd(i):
    s = salts[i % len(salts)]; line.set_data(r, U_DH(r, s/1000))
    txt.set_text(f"salt = {s} mM\nDebye length 1/κ = {1/kappa(max(s,1e-9)/1000):.2f} nm")
    ax.set_title("Debye–Hückel: salt screens the charge"); return line, txt
anim = animation.FuncAnimation(fig, upd, frames=len(salts), interval=600, blit=True)
plt.close(fig); HTML(anim.to_jshtml())

---
# Part 2 — Setting up the V21 simulation (with visualisations)

We now configure the V21 system the way CALVADOS expects. **We will not run a long simulation here** (that needs a GPU cluster) — instead we build every input and *look at what we've made* at each step. First, install the lightweight pieces we need (CALVADOS's configuration module + a 3-D viewer + mdtraj) and clone the repo's trajectory data.

> *Install note:* we don't run MD in this notebook, so we install **only** what's needed to *configure* and *analyse*. CALVADOS's package `__init__` eagerly imports a heavy analysis stack (MDAnalysis, numba, localcider, …) that doesn't install on Colab's Python — but the part we use, `calvados.cfg`, is self-contained, so we trim that `__init__` after install. (To actually *run* simulations you'd additionally install OpenMM and use a GPU.)

In [ ]:
import sys, subprocess, os, importlib, importlib.util, pathlib
def pip(*a): subprocess.run([sys.executable,"-m","pip","install","-q",*a], check=True)

# light deps only (no MD engine / no MDAnalysis stack — we configure + analyse, not run, here)
pip("mdtraj", "py3Dmol", "Jinja2", "pyyaml")
pip("--no-deps", "git+https://github.com/KULL-Centre/CALVADOS.git@v0.7.0")

# CALVADOS __init__ eagerly imports calvados.analysis/.sequence -> MDAnalysis, numba, localcider,
# BLOCKING, and pins numpy==1.24 (no Colab/py3.12 wheel). We only use the self-contained
# calvados.cfg (Config/Components), so trim __init__ to skip those heavy imports.
importlib.invalidate_caches()
pkg = pathlib.Path(importlib.util.find_spec("calvados").origin).parent
(pkg / "__init__.py").write_text("# trimmed for this tutorial: only calvados.cfg is used\n")

from calvados.cfg import Config, Components            # the only CALVADOS API we need
import mdtraj as md, py3Dmol
print("calvados.cfg + mdtraj", md.version.version, "+ py3Dmol ready")

# pre-simulated trajectories shipped in this repo
if not os.path.isdir("V21-RNA-simulation-CALVADOS-tutorial"):
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/anshuJS/V21-RNA-simulation-CALVADOS-tutorial.git"], check=True)
DATA = "V21-RNA-simulation-CALVADOS-tutorial/data"
print("data:", sorted(os.listdir(DATA)))

## Step 1 — the RNA and the 2-bead model

V21 is 21 nucleotides → **42 beads** (a phosphate + a base bead per nucleotide). Two beads per residue, ordered phosphate-then-base, so for residue $p$ (1-based): phosphate bead $=2p-1$, **base bead $=2p$**. That base-bead index is what we'll restrain to make the hairpin.

In [ ]:
SEQ        = "UAUAAUUAAUAACUAAUUACU"
NRES       = len(SEQ)                                  # 21
STEM_PAIRS = [(3,19),(4,18),(5,17),(6,16),(7,15),(8,14)]   # RNAfold hairpin stem
base_bead  = lambda p: 2*p
print(f"{SEQ}  ->  {NRES} nt = {2*NRES} beads")
print("hairpin stem base-bead pairs:", [(base_bead(i), base_bead(j)) for i,j in STEM_PAIRS])

## Step 2 — bead parameters & the generic-`r` decision

CALVADOS has **no cytosine/guanine** RNA parameters (every base bead is `a`, `u`, or generic `r`; only size differs). Because V21 contains C, and because base identity barely matters in this model, we represent **every nucleotide with the generic `r` bead**. The FASTA we feed CALVADOS is therefore 21 `r`'s.

In [ ]:
import os
os.makedirs("V21_setup", exist_ok=True)
RES_CSV = """three,one,MW,lambdas,sigmas,q,bondlength
ARG,R,156.19,0.7307624767517166,0.6559999999999999,1,0.38
ASP,D,115.09,0.0416040480605567,0.5579999999999999,-1,0.38
ASN,N,114.1,0.4255859009787713,0.568,0,0.38
GLU,E,129.11,0.0006935460962935,0.5920000000000001,-1,0.38
LYS,K,128.17,0.1790211738990582,0.636,1,0.38
HIS,H,137.14,0.4663667290557992,0.608,0,0.38
GLN,Q,128.13,0.3934318551056041,0.602,0,0.38
SER,S,87.08,0.4625416811611541,0.518,0,0.38
CYS,C,103.14,0.5615435099141777,0.5479999999999999,0,0.38
GLY,G,57.05,0.7058843733666401,0.45,0,0.38
THR,T,101.11,0.3713162976273964,0.562,0,0.38
ALA,A,71.07,0.2743297969040348,0.504,0,0.38
MET,M,131.2,0.5308481134337497,0.618,0,0.38
TYR,Y,163.18,0.9774611449343455,0.6459999999999999,0,0.38
VAL,V,99.13,0.2083769608174481,0.5860000000000001,0,0.38
TRP,W,186.22,0.9893764740371644,0.6779999999999999,0,0.38
LEU,L,113.16,0.6440005007782226,0.618,0,0.38
ILE,I,113.16,0.5423623610671892,0.618,0,0.38
PRO,P,97.12,0.3593126576364644,0.5559999999999999,0,0.38
PHE,F,147.18,0.8672358982062975,0.636,0,0.38
LPH,B,100.0,0.4625416811611541,0.855,0,0.38
LPT,Z,200.0,0.9774611449343455,0.9,0,0.38
ADE,a,134.1,1.18,0.6430,0,0.54
URI,u,111.1,1.18,0.5958,0,0.54
RBC,p,194.1,0.00,0.6954,-1,0.59
RNA,r,126.3,1.18,0.6238,0,0.54"""
open("V21_setup/residues_C2RNA.csv","w").write(RES_CSV)
open("V21_setup/rna.fasta","w").write(">V21\n" + "r"*NRES + "\n")
print("RNA base beads available (note: no c/g):")
for line in RES_CSV.splitlines():
    p = line.split(",")
    if len(p)>1 and p[1] in ("a","u","r","p"): print("  ", line)

## Step 3 — write `prepare.py` and generate the inputs

CALVADOS builds a simulation from a small `prepare.py` that defines a `Config` (run settings) and `Components` (the molecule), then writes `config.yaml`, `components.yaml`, and `run.py`. We set 150 mM salt (`ionic=0.15`) and **`platform='CPU'`**. (We won't launch a long run — we just need the generated topology to visualise.)

In [ ]:
PREP = r"""
import os, subprocess
from calvados.cfg import Config, Components
cwd=os.getcwd(); sysname='V21'
config=Config(sysname=sysname, box=[40,40,40], temp=300, ionic=0.15, pH=7.0,
    topol='center', wfreq=1000, steps=2000, runtime=0, platform='CPU',
    restart='checkpoint', frestart='restart.chk', verbose=True{CRES})
path=f'{cwd}/{sysname}'; subprocess.run(f'mkdir -p {path}',shell=True); subprocess.run('mkdir -p data',shell=True)
config.write(path, name='config.yaml')
c=Components(); c.add(name='V21', molecule_type='rna', restraint=False, charge_termini='both',
    fresidues=f'{cwd}/residues_C2RNA.csv', ffasta=f'{cwd}/rna.fasta', nmol=1,
    rna_kb1=1400.0, rna_kb2=2200.0, rna_ka=4.20, rna_pa=3.14,
    rna_nb_sigma=0.4, rna_nb_scale=15, rna_nb_cutoff=2.0)
c.write(path, name='components.yaml')
"""
open("V21_setup/prepare.py","w").write(PREP.replace("{CRES}",""))
import subprocess, sys
r = subprocess.run([sys.executable,"prepare.py"], cwd="V21_setup",
                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(r.stdout[-400:] if r.stdout else "", "\nrun.py generated:",
      os.path.exists("V21_setup/V21/run.py"))

## Step 4 — **visualise the system in 3-D**

Let's actually look at the 42-bead V21 chain. We render a real CALVADOS snapshot (from the shipped trajectories) with `py3Dmol`: **orange = phosphate beads, green = base beads**. This is the molecule you just configured.

In [ ]:
import py3Dmol, mdtraj as md
def show_beads(dcd, top, frame=0, pairs=None, title=""):
    t = md.load_frame(dcd, frame, top=top)
    t.save_pdb("/tmp/_frame.pdb"); pdb = open("/tmp/_frame.pdb").read()
    v = py3Dmol.view(width=520, height=420)
    v.addModel(pdb, "pdb")
    v.setStyle({"elem":"P"}, {"sphere":{"color":"orange", "radius":3.2}})   # phosphate
    v.setStyle({"elem":"N"}, {"sphere":{"color":"forestgreen","radius":3.2}})# base
    if pairs:                                            # draw restraint bonds
        xyz = t.xyz[0]*10.0                              # nm -> Angstrom
        for i,j in pairs:
            a, b = xyz[2*i-1], xyz[2*j-1]                # base beads (0-based 2p-1)
            v.addCylinder({"start":{"x":float(a[0]),"y":float(a[1]),"z":float(a[2])},
                           "end":{"x":float(b[0]),"y":float(b[1]),"z":float(b[2])},
                           "radius":0.6,"color":"red","dashed":True})
    v.zoomTo()
    if title: v.addLabel(title, {"fontSize":12,"backgroundColor":"white","fontColor":"black"})
    return v.show()

# an extended disordered snapshot
show_beads(f"{DATA}/trajectories/disordered/V21_dis_0mM/traj.dcd",
           f"{DATA}/trajectories/disordered/V21_dis_0mM/top.pdb", frame=0,
           title="V21 (disordered, 0 mM) — orange=phosphate, green=base")

## Step 5 — the hairpin: adding base-pair restraints

The model can't fold the hairpin on its own, so we add **6 harmonic restraints** between the base beads of the stem pairs, via CALVADOS *custom restraints* (`cres`). The file format is `name copy bead | name copy bead | r k`; we use $r=0.62$ nm and $k=700$ kJ/mol/nm². Below we write the restraints **and visualise them** (red dashed bonds) on a *folded* snapshot — you can see the 6 pairs holding the stem closed.

In [ ]:
os.makedirs("V21_setup_hp/input", exist_ok=True)
with open("V21_setup_hp/input/cres.txt","w") as f:
    for i,j in STEM_PAIRS:
        f.write(f"V21 1 {base_bead(i)} | V21 1 {base_bead(j)} | 0.6238 700.0\n")
print(open("V21_setup_hp/input/cres.txt").read())

# visualise restraints on a folded hairpin snapshot from the shipped data
show_beads(f"{DATA}/trajectories/hairpin/V21_hp_150mM/traj.dcd",
           f"{DATA}/trajectories/hairpin/V21_hp_150mM/top.pdb", frame=500,
           pairs=STEM_PAIRS, title="V21 hairpin — red dashed = the 6 stem restraints")

That's the whole setup: a generic-`r` 2-bead RNA, optionally pinned into a hairpin by 6 restraints, in a 40 nm box at a chosen salt and temperature. To run it *for real* you'd set `platform='CUDA'` and `steps` to ~$10^8$ (1 µs) and launch on a GPU. We've already done that for you — on to the analysis.

---
# Part 3 — Analysing pre-simulated trajectories

The repo ships **1 µs CALVADOS runs of V21 at 12 salt concentrations** (0–1000 mM), for both the disordered and hairpin variants (down-sampled to 1000 frames each for size). We'll measure two classic shape descriptors and see how salt reshapes the RNA.

## Step 6 — radius of gyration and end-to-end distance

- **Radius of gyration** $R_g$ — overall size (how spread out the beads are):
  $$R_g=\sqrt{\tfrac{1}{N}\sum_i \lVert \mathbf{r}_i-\mathbf{r}_\text{com}\rVert^2}$$
- **End-to-end distance** — straight-line distance between the first and last nucleotide; we report the root-mean-square, $\sqrt{\langle R_{ee}^2\rangle}$.

One subtlety: the molecule can wrap across the periodic box, so we **unwrap** it first (follow the chain bead-by-bead using the minimum-image convention) before measuring.

In [ ]:
import numpy as np, mdtraj as md
def e2e_rg(dcd, top, equil=0.1):
    t = md.load(dcd, top=top)
    xyz = t.xyz.astype(np.float64); box = t.unitcell_lengths.astype(np.float64)
    F, nb, _ = xyz.shape; nres = nb//2
    d = xyz[:,1:,:] - xyz[:,:-1,:]; d -= np.round(d/box[:,None,:])*box[:,None,:]   # min-image
    uw = np.empty_like(xyz); uw[:,0,:] = xyz[:,0,:]; uw[:,1:,:] = xyz[:,:1,:] + np.cumsum(d,axis=1)
    cent = uw.reshape(F,nres,2,3).mean(axis=2)                                     # residue centroids
    e2e = np.linalg.norm(cent[:,-1,:]-cent[:,0,:], axis=-1)
    rg  = np.sqrt(((cent-cent.mean(1,keepdims=True))**2).sum(-1).mean(-1))
    n0 = int(F*equil)
    return e2e[n0:], rg[n0:]                                                       # drop equilibration

# demo at 150 mM, both variants
for v,pre in [("disordered","dis"),("hairpin","hp")]:
    e,r = e2e_rg(f"{DATA}/trajectories/{v}/V21_{pre}_150mM/traj.dcd",
                 f"{DATA}/trajectories/{v}/V21_{pre}_150mM/top.pdb")
    print(f"{v:11s} 150 mM:  RMS e2e = {np.sqrt((e**2).mean()):.2f} nm   <Rg> = {r.mean():.2f} nm")

## Step 7 — distributions at 150 mM (disordered vs hairpin)

The hairpin is far more compact and narrowly distributed (the restraints pin it); the disordered chain samples a broad range of extended shapes.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for v,pre,c in [("disordered","dis","#1f77b4"),("hairpin","hp","#d62728")]:
    e,r = e2e_rg(f"{DATA}/trajectories/{v}/V21_{pre}_150mM/traj.dcd",
                 f"{DATA}/trajectories/{v}/V21_{pre}_150mM/top.pdb")
    ax[0].hist(e, bins=40, density=True, alpha=0.6, color=c, label=v)
    ax[1].hist(r, bins=40, density=True, alpha=0.6, color=c, label=v)
ax[0].set_xlabel("end-to-end distance (nm)"); ax[1].set_xlabel("$R_g$ (nm)")
ax[0].set_ylabel("probability density"); ax[0].legend(); ax[1].legend()
fig.suptitle("V21 at 150 mM — shape distributions (1 µs runs)"); plt.tight_layout(); plt.show()

## Step 8 — the salt titration: dimensions vs salt

Now the full series. As salt rises, Debye screening (the Part-1 animation!) weakens the phosphate–phosphate repulsion, so the **disordered** chain **compacts**. The **hairpin** is held by restraints, so it stays compact and barely moves — bracketing the two structural extremes.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
SALTS = [0,10,20,50,100,150,200,300,400,500,700,1000]
fig, ax = plt.subplots(2, 1, figsize=(7, 8), sharex=True)
for v,pre,c in [("disordered","dis","#1f77b4"),("hairpin","hp","#d62728")]:
    rms, rg = [], []
    for s in SALTS:
        e,r = e2e_rg(f"{DATA}/trajectories/{v}/V21_{pre}_{s}mM/traj.dcd",
                     f"{DATA}/trajectories/{v}/V21_{pre}_{s}mM/top.pdb")
        rms.append(np.sqrt((e**2).mean())); rg.append(r.mean())
    ax[0].plot(SALTS, rms, "-o", color=c, label=v)
    ax[1].plot(SALTS, rg,  "-o", color=c, label=v)
ax[0].set_ylabel("RMS end-to-end (nm)"); ax[1].set_ylabel("$R_g$ (nm)")
ax[1].set_xlabel("salt concentration (mM)")
ax[0].legend(); ax[1].legend(); ax[0].set_title("V21 RNA: salt-induced compaction")
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Step 9 — see it directly: extended vs compact

Finally, look at the two extremes in 3-D: the disordered chain at **0 mM** (extended) vs **1000 mM** (compact). This is the salt effect you just quantified, made visible.

In [ ]:
from IPython.display import display
print("disordered, 0 mM (low salt → extended):")
display(show_beads(f"{DATA}/trajectories/disordered/V21_dis_0mM/traj.dcd",
                   f"{DATA}/trajectories/disordered/V21_dis_0mM/top.pdb", frame=900))
print("disordered, 1000 mM (high salt → compact):")
display(show_beads(f"{DATA}/trajectories/disordered/V21_dis_1000mM/traj.dcd",
                   f"{DATA}/trajectories/disordered/V21_dis_1000mM/top.pdb", frame=900))

---
# Wrap-up

You went from first principles to a real result:
- **MD** integrates $\mathbf{F}=-\nabla U$; **coarse-graining** makes long timescales reachable; **CALVADOS** sums simple bond/angle/Ashbaugh–Hatch/Debye–Hückel terms.
- You **configured** a 2-bead CG RNA and visualised it, and saw how a **hairpin** is imposed with 6 restraints.
- You **analysed** 1 µs runs and reproduced the physics: **salt screens charge → disordered RNA compacts** ($R_{ee}$ ~7→5 nm), while the **restrained hairpin stays pinned**.

**Where to go next**
- Run your own: set `platform='CUDA'`, `steps≈1e8`, loop over salts/temperatures, on a GPU.
- A more faithful hairpin would use CALVADOS v0.8's elastic-network restraints (a 3-D reference structure) instead of 6 base-pair bonds.
- Try the *temperature* axis, or different sequences/lengths.

*Data and code: this repository. CALVADOS: Tesei et al., and the KULL-Centre/CALVADOS GitHub.*
